In [11]:
import numpy as np
import pandas as pd
df = pd.read_csv('T7Dataset.csv')
print(df.columns)
df["decision"] = df["decision"].map({True:1, False:0})

Index(['Id', 'gender', 'age', 'nationality', 'sport', 'ind-university_grade',
       'ind-debateclub', 'ind-programming_exp', 'ind-international_exp',
       'ind-entrepeneur_exp', 'ind-languages', 'ind-exact_study', 'ind-degree',
       'company', 'decision'],
      dtype='object')


In [12]:
def age_group(age):
    if age < 25:
        return "<25"
    elif age <= 35:
        return "25-35"
    else:
        return "35+"

df['age_group'] = df['age'].apply(age_group)

df = df.dropna() 
df = df.drop(columns=["Id"])
df = df.drop(columns=["age"])
df = df.drop(columns=["ind-languages"])

In [13]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['decision'])
y = df['decision']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [14]:
def entropy(y):
    proportions = y.value_counts(normalize=True)
    return -sum(p * np.log2(p) for p in proportions if p > 0)

In [15]:
def information_gain(X, y, feature):
    total_entropy = entropy(y)
    
    weighted_entropy = 0
    for value in X[feature].unique():
        subset_y = y[X[feature] == value]
        weight = len(subset_y) / len(y)
        weighted_entropy += weight * entropy(subset_y)
    
    return total_entropy - weighted_entropy

In [16]:
def id3(X, y, depth=0, max_depth=4):
    
    # If all labels are same → return leaf
    if len(y.unique()) == 1:
        return y.iloc[0]
    
    # Stop condition
    if depth == max_depth or len(X.columns) == 0:
        return y.mode()[0]
    
    # Find best feature
    gains = {col: information_gain(X, y, col) for col in X.columns}
    best_feature = max(gains, key=gains.get)
    
    tree = {best_feature: {}}
    
    # Create branches
    for value in X[best_feature].unique():
        subset_X = X[X[best_feature] == value].drop(columns=[best_feature])
        subset_y = y[X[best_feature] == value]
        
        if len(subset_y) == 0:
            tree[best_feature][value] = y.mode()[0]
        else:
            tree[best_feature][value] = id3(
                subset_X,
                subset_y,
                depth+1,
                max_depth
            )
    
    return tree

In [17]:
tree = id3(X_train, y_train, max_depth=4)

print(tree)

{'ind-degree': {'master': {'ind-university_grade': {np.int64(73): {'company': {'B': {'ind-entrepeneur_exp': {np.True_: np.int64(1), np.False_: np.int64(0)}}, 'C': np.int64(0), 'D': np.int64(1), 'A': {'sport': {'Chess': np.int64(1), 'Running': np.int64(0), 'Tennis': np.int64(1), 'Swimming': np.int64(1), 'Football': np.int64(1)}}}}, np.int64(69): {'company': {'D': {'nationality': {'Dutch': np.int64(1), 'Belgian': np.int64(1), 'German': np.int64(1)}}, 'C': {'sport': {'Swimming': np.int64(0), 'Golf': np.int64(0), 'Chess': np.int64(0), 'Cricket': np.int64(1), 'Rugby': np.int64(0), 'Football': np.int64(0), 'Tennis': np.int64(0)}}, 'A': {'sport': {'Rugby': np.int64(0), 'Tennis': np.int64(1), 'Chess': np.int64(0), 'Football': np.int64(0), 'Swimming': np.int64(0), 'Golf': np.int64(1), 'Cricket': np.int64(1), 'Running': np.int64(0)}}, 'B': {'ind-entrepeneur_exp': {np.False_: np.int64(0), np.True_: np.int64(1)}}}}, np.int64(70): {'company': {'C': {'sport': {'Swimming': np.int64(0), 'Running': np.

In [18]:
def predict_one(tree, sample):
    if not isinstance(tree, dict):
        return tree
    
    feature = next(iter(tree))
    value = sample[feature]
    
    if value in tree[feature]:
        return predict_one(tree[feature][value], sample)
    else:
        return y_train.mode()[0]  # fallback

def predict(tree, X):
    return X.apply(lambda row: predict_one(tree, row), axis=1)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = predict(tree, X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.74875
Precision: 0.6494252873563219
Recall: 0.44664031620553357
F1: 0.5292740046838408


In [20]:
df_test = X_test.copy()
df_test["actual"] = y_test
df_test["predicted"] = y_pred
print(df_test.groupby("gender")["predicted"].mean())
def true_positive_rate(group):
    positives = group[group["actual"] == 1]
    if len(positives) == 0:
        return 0
    return (positives["predicted"] == 1).mean()

print(df_test.groupby("gender").apply(true_positive_rate))
print(df_test.groupby("age_group")["predicted"].mean())
print(df_test.groupby("age_group").apply(true_positive_rate))

gender
female    0.193460
male      0.244019
other     0.066667
Name: predicted, dtype: float64
gender
female    0.401961
male      0.480000
other     0.000000
dtype: float64
age_group
25-35    0.225177
<25      0.199153
Name: predicted, dtype: float64
age_group
25-35    0.470270
<25      0.382353
dtype: float64


C:\Users\user\AppData\Local\Temp\ipykernel_15052\968371153.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_test.groupby("gender").apply(true_positive_rate))
C:\Users\user\AppData\Local\Temp\ipykernel_15052\968371153.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_test.groupby("age_group").apply(true_positive_rate))
